# KServe Inference Protocol Versions: v1 vs v2

KServe supports two inference protocols for serving ML models:

- **v1** — the original _predict API_, a simpler row-major format
- **v2** — the _Open Inference Protocol (OIP)_, a standardised, feature-rich format adopted from the KServe/Triton community

This notebook walks through the **practical differences** between the two protocols using a real model — the **Mobile Price Classification SVM** stored in MLflow.

> The model is trained in [../../mlflow/mobile-price-classification/](../../mlflow/mobile-price-classification/).  
> It uses 20 normalised features to classify phones into 4 price ranges (0–3).

In [ ]:
import json
from pathlib import Path
from IPython.display import Markdown, display

---

## 1. InferenceService: v1 vs v2

Both protocols use **the same model artifact** in MLflow and the same `mlflow://` storage URI.
However, the manifests differ in **two fields**: `modelFormat` and `protocolVersion`.

> **Before applying the manifests**, replace all `<>` placeholder values in `v1_InferenceService.yaml` and `v2_InferenceService.yaml` with your own:
> - `<inference-name>` — a name of your choice for the InferenceService
> - `<workspace-name>` — your Kubernetes namespace / workspace name
> - `<username>` — the username used when registering the model in MLflow

Let's compare the two YAML files side-by-side.

In [ ]:
v1_yaml = Path("v1_InferenceService.yaml").read_text()
v2_yaml = Path("v2_InferenceService.yaml").read_text()

display(Markdown(
    "| v1 InferenceService | v2 InferenceService |\n"
    "|---|---|\n"
    "| <pre>" + v1_yaml.replace("\n", "<br>") + "</pre>"
    " | <pre>" + v2_yaml.replace("\n", "<br>") + "</pre> |"
))

The manifests differ in two fields:

| Field | v1 | v2 |
|---|---|---|
| `spec.predictor.model.modelFormat.name` | `sklearn` | `mlflow` |
| `spec.predictor.model.protocolVersion` | `v1` | `v2` |

Both use the same `storageUri: mlflow://...` — the MLflow storage initializer fetches and unpacks the model artifact regardless of which runtime serves it.

### Why does the model format differ?

KServe auto-selects a **serving runtime** based on the `modelFormat` and `protocolVersion` you specify. Not every runtime supports every combination:

| Runtime | Supported model formats | Supported protocol versions |
|---|---|---|
| `kserve-mlserver` | `mlflow`, `sklearn`, `xgboost`, `lightgbm` | **v2 only** |
| `kserve-sklearnserver` | `sklearn` | **v1 and v2** |

- **v2** can use `modelFormat: mlflow` because `kserve-mlserver` supports both the MLflow model wrapper and the v2 protocol.
- **v1** must use `modelFormat: sklearn` because `kserve-sklearnserver` is the only runtime that speaks v1 for sklearn-based models.

If you try `modelFormat: mlflow` with `protocolVersion: v1`, KServe will reject the InferenceService with:

```
Warning  InternalError  no runtime found to support predictor with model type: {mlflow <nil>}
```

This happens because no built-in `ClusterServingRuntime` advertises support for both `mlflow` format and `v1` protocol.

---

## 2. Request Format: v1 (Predict API)

The v1 protocol uses a **row-major** layout. Each sample is a flat list of feature values, and all samples are collected under the `instances` key.

```
POST /v1/models/<model-name>:predict
```

```json
{
  "instances": [
    [0.36, 1.0, 0.52, ...],   // sample 1 (20 floats)
    [0.23, 1.0, 0.00, ...],   // sample 2
    ...
  ]
}
```

The response is equally simple:

```json
{
  "predictions": [3, 3, 2, 3, 1]
}
```

Features are **positional** — there are no column names. The server expects values in the same order the model was trained on.

In [ ]:
with open("v1-mlflow-inference-body.json") as f:
    v1_body = json.load(f)

print(f"Number of samples : {len(v1_body['instances'])}")
print(f"Features per sample: {len(v1_body['instances'][0])}")
print()
print("First sample (flat list of 20 floats):")
print(json.dumps(v1_body["instances"][0], indent=2))

---

## 3. Request Format: v2 (Open Inference Protocol)

The v2 protocol uses a **column-major** (tensor) layout. Each feature becomes a named input tensor with explicit `shape` and `datatype` metadata.

```
POST /v2/models/<model-name>/infer
```

```json
{
  "inputs": [
    {"name": "battery_power", "shape": [5], "datatype": "FP64", "data": [0.36, 0.23, ...]},
    {"name": "blue",          "shape": [5], "datatype": "FP64", "data": [1.0, 1.0, ...]},
    ...
  ]
}
```

The response mirrors this structure:

```json
{
  "model_name": "...",
  "model_version": "1",
  "outputs": [
    {"name": "output-0", "shape": [5], "datatype": "INT64", "data": [3, 3, 2, 3, 1]}
  ]
}
```

**Key advantages of v2:**
- Features are **named** — no ambiguity about column order
- **Datatypes** are explicit (`FP64`, `INT64`, `BYTES`, etc.)
- Dedicated **metadata** (`/v2/models/<name>`) and **readiness** (`/v2/models/<name>/ready`) endpoints
- A standardised specification shared across KServe, Triton, Seldon, and others

In [ ]:
with open("v2-mlflow-inference-body.json") as f:
    v2_body = json.load(f)

print(f"Number of input tensors (features): {len(v2_body['inputs'])}")
print(f"Samples per tensor               : {v2_body['inputs'][0]['shape'][0]}")
print()
print("First 3 input tensors:")
print(json.dumps(v2_body["inputs"][:3], indent=2))

---

## 4. Converting Between v1 and v2

The data is the same — only the **layout** differs.  Think of it as transposing a matrix:

```
v1  (row-major):     samples x features   →  [[s1_f1, s1_f2, ...], [s2_f1, s2_f2, ...]]
v2  (column-major):  features x samples   →  [{name: f1, data: [s1_f1, s2_f1, ...]}, ...]
```

### 4a. v1 → v2 Conversion

We need the feature names (the v1 format doesn't carry them) and then transpose rows into columns.

In [ ]:
FEATURE_NAMES = [
    "battery_power", "blue", "clock_speed", "dual_sim", "fc",
    "four_g", "int_memory", "m_dep", "mobile_wt", "n_cores",
    "pc", "px_height", "px_width", "ram", "sc_h",
    "sc_w", "talk_time", "three_g", "touch_screen", "wifi",
]


def v1_to_v2(v1_body: dict, feature_names: list[str] = FEATURE_NAMES) -> dict:
    """Convert a v1 request body (row-major) to v2 (column-major tensors)."""
    rows = v1_body["instances"]
    n_samples = len(rows)
    inputs = []
    for col_idx, name in enumerate(feature_names):
        inputs.append({
            "name": name,
            "shape": [n_samples],
            "datatype": "FP64",
            "data": [rows[row_idx][col_idx] for row_idx in range(n_samples)],
        })
    return {"inputs": inputs}

In [ ]:
converted_v2 = v1_to_v2(v1_body)

print("Converted v2 body (first 3 tensors):")
print(json.dumps(converted_v2["inputs"][:3], indent=2))

### 4b. v2 → v1 Conversion

The reverse: collapse named column tensors back into flat row lists.

In [ ]:
def v2_to_v1(v2_body: dict) -> dict:
    """Convert a v2 request body (column-major tensors) to v1 (row-major)."""
    inputs = v2_body["inputs"]
    n_samples = inputs[0]["shape"][0]
    n_features = len(inputs)
    rows = []
    for row_idx in range(n_samples):
        rows.append([inputs[col_idx]["data"][row_idx] for col_idx in range(n_features)])
    return {"instances": rows}

In [ ]:
converted_v1 = v2_to_v1(v2_body)

print("Converted v1 body (first sample):")
print(json.dumps(converted_v1["instances"][0], indent=2))

---

## 5. Quick Reference

| | v1 (Predict API) | v2 (Open Inference Protocol) |
|---|---|---|
| **Model format** | `sklearn` | `mlflow` |
| **Serving runtime** | `kserve-sklearnserver` | `kserve-mlserver` |
| **Inference endpoint** | `POST /v1/models/<name>:predict` | `POST /v2/models/<name>/infer` |
| **Request layout** | Row-major (`instances`) | Column-major named tensors (`inputs`) |
| **Feature names** | Not included (positional) | Included in each tensor (`name`) |
| **Data types** | Implicit | Explicit (`datatype` field) |
| **Response key** | `predictions` | `outputs` (with `name`, `shape`, `datatype`) |
| **Metadata endpoint** | `/v1/models/<name>` (basic) | `/v2/models/<name>` (detailed) |
| **Readiness endpoint** | `/v1/models/<name>` | `/v2/models/<name>/ready` |
| **Spec** | KServe-specific | Standardised across KServe, Triton, Seldon |

---

## 6. Live Testing (Optional)

If you have deployed both InferenceServices, you can test them here.  
Fill in your connection details and run the cells below.

In [ ]:
import time
import requests

# ---------------------------------------------------------------------------
# Connection settings  (fill these in)
# ---------------------------------------------------------------------------
INFERENCE_SERVICE_URI_V1 = ""  # external URL, no trailing slash (copied from KServe InferenceService status.url)
INFERENCE_SERVICE_URI_V2 = ""  # external URL, no trailing slash (copied from KServe InferenceService status.url)
API_KEY = ""  # API key, provided by your administrator 

# ---------------------------------------------------------------------------
# Model names  (must match metadata.name in the InferenceService YAMLs)
# ---------------------------------------------------------------------------
MODEL_NAME_V1 = "v1-mobile-price-classification-inference"  # from v1_InferenceService.yaml
MODEL_NAME_V2 = "v2-mobile-price-classification-inference"  # from v2_InferenceService.yaml

HEADERS = {"X-Api-Key": API_KEY, "Content-Type": "application/json"}

if not INFERENCE_SERVICE_URI_V1 or not INFERENCE_SERVICE_URI_V2 or not API_KEY:
    raise ValueError("Please fill in the connection settings (URIs and API key) before running the tests.")

print(f"v1 URI : {INFERENCE_SERVICE_URI_V1}")
print(f"v2 URI : {INFERENCE_SERVICE_URI_V2}")
print(f"v1 model : {MODEL_NAME_V1}")
print(f"v2 model : {MODEL_NAME_V2}")

### 6a. Health & Readiness Checks

In [ ]:
def check_ready(protocol: str, inference_name: str):
    """Check if the model is ready to serve."""
    if protocol == "v2":
        url = f"{INFERENCE_SERVICE_URI_V2}/v2/models/{inference_name}/ready"
    else:
        url = f"{INFERENCE_SERVICE_URI_V1}/v1/models/{inference_name}"

    print(f"GET {url}")
    resp = requests.get(url, headers=HEADERS)
    print(f"Status: {resp.status_code}")
    try:
        print(json.dumps(resp.json(), indent=2))
    except Exception:
        print(resp.text)


print("--- v1 readiness ---")
check_ready("v1", MODEL_NAME_V1)
print()
print("--- v2 readiness ---")
check_ready("v2", MODEL_NAME_V2)

### 6b. Send Inference Requests

In [ ]:
def send_inference(body: dict, protocol: str, inference_name: str) -> requests.Response:
    """Send a POST inference request and print the result with timing."""

    if protocol not in ("v1", "v2"):
        raise ValueError(f"Invalid protocol: {protocol}. Must be 'v1' or 'v2'.")

    if protocol == "v2":
        url = f"{INFERENCE_SERVICE_URI_V2}/v2/models/{inference_name}/infer"
    else:
        url = f"{INFERENCE_SERVICE_URI_V1}/v1/models/{inference_name}:predict"

    print(f"POST {url}")
    start = time.perf_counter()
    resp = requests.post(url, headers=HEADERS, json=body)
    elapsed_ms = (time.perf_counter() - start) * 1000

    print(f"Status : {resp.status_code}")
    print(f"Latency: {elapsed_ms:.1f} ms")
    try:
        print(json.dumps(resp.json(), indent=2))
    except Exception:
        print(resp.text)
    return resp

In [ ]:
print("=" * 60)
print("v1 (Predict API)")
print("=" * 60)
resp_v1 = send_inference(v1_body, "v1", MODEL_NAME_V1)

print()
print("=" * 60)
print("v2 (Open Inference Protocol)")
print("=" * 60)
resp_v2 = send_inference(v2_body, "v2", MODEL_NAME_V2)

### 6c. Compare Predictions

Both protocols serve the same model, so the predicted classes should be identical.

In [ ]:
try:
    preds_v1 = resp_v1.json()["predictions"]
    preds_v2 = resp_v2.json()["outputs"][0]["data"]

    print(f"v1 predictions: {preds_v1}")
    print(f"v2 predictions: {preds_v2}")
    print()
    if preds_v1 == preds_v2:
        print("Predictions match — same model, different protocol.")
    else:
        print("Predictions differ — check model versions or input data.")
except Exception as e:
    print(f"Could not compare (did both requests succeed?): {e}")